In [33]:
!pip uninstall -y tensorflow tensorflow-cpu tensorflow-intel tensorflow-metadata protobuf
!pip install -q tensorflow==2.15.0 tensorflow-text==2.15.0
!pip install -q protobuf==4.25.3
!pip install -q numpy pandas scikit-learn matplotlib seaborn tqdm category_encoders
!pip install -q xgboost lightgbm catboost

print("✔ Environment ready with TensorFlow GPU 2.15")

Found existing installation: protobuf 4.25.3
Uninstalling protobuf-4.25.3:
  Successfully uninstalled protobuf-4.25.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-decision-forests 1.11.0 requires ydf, which is not installed.
tensorflow-datasets 4.9.9 requires tensorflow-metadata, which is not installed.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
dopamine-rl 4.1.2 requires tf-keras>=2.18.0, but you have tf-keras 2.15.1 which is incompatible.
pydrive2 1.21.3 requires cryptography<44, but you have cryptography 46.0.3 which is incompatible.
pydrive2 1.21.3 requires pyOpenSSL<=24.2.1,>=19.1.0, but you have pyopenssl 25.3.0 which is incompatible.
tensorflow-decision-forests 1.11.0 requires tensorflow==2.18.0, but you have tensorflow 2.15.0 which is incompatible.
tensorflow-decision-forests 1.11.0 re

In [34]:
# ================================
# Silence TF logs
# ================================
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'  # Suppress CUDA/TF warnings

# ================================
# Core Libraries
# ================================
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

# ================================
# Visualization
# ================================
import matplotlib.pyplot as plt
import seaborn as sns

# ================================
# Progress Bar
# ================================
from tqdm.notebook import tqdm

# ================================
# Data Encoding
# ================================
import category_encoders as ce

# ================================
# TensorFlow (GPU 2.15)
# ================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, Dropout, BatchNormalization, Activation, Input
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

# ================================
# Sklearn
# ================================
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV

# ================================
# Gradient Boosting Models
# ================================
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool

print("✔ All imports loaded successfully.")

✔ All imports loaded successfully.


In [35]:
train_df = pd.read_csv('/kaggle/input/playground-series-s5e11/train.csv')
test_df = pd.read_csv('/kaggle/input/playground-series-s5e11/test.csv')
submission = pd.read_csv('/kaggle/input/playground-series-s5e11/sample_submission.csv')

In [36]:
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Configuration
FOLDS = 10
SEED = 42
TARGET = 'loan_paid_back'

In [37]:
# Drop ID columns
if 'id' in train_df.columns:
    train_df = train_df.drop(columns=['id'])
    test_df = test_df.drop(columns=['id'])

In [38]:
# Separate Target
y = train_df[TARGET]
X = train_df.drop(columns=[TARGET])
X_test = test_df.copy()

# Combine for consistent preprocessing
train_len = len(X)
full_df = pd.concat([X, X_test], axis=0).reset_index(drop=True)

In [39]:
# Handle Missing Values (Simple Median/Mode strategy for Trees)
num_cols = full_df.select_dtypes(exclude=['object', 'category']).columns.tolist()
cat_cols = full_df.select_dtypes(include=['object', 'category']).columns.tolist()

for col in num_cols:
    full_df[col] = full_df[col].fillna(full_df[col].median())

for col in cat_cols:
    full_df[col] = full_df[col].fillna("Missing").astype(str)

print(f"Data Loaded. Train Shape: {X.shape}, Test Shape: {X_test.shape}")

Data Loaded. Train Shape: (593994, 11), Test Shape: (254569, 11)


In [40]:
def create_features(df):
    df['loan_to_income'] = df['loan_amount'] / (df['annual_income'] + 1)
    df['monthly_burden'] = df['loan_amount'] / (df['annual_income'] / 12 + 1)
    df['interest_burden'] = df['loan_amount'] * (df['interest_rate'] / 100)
    
    df['loan_vs_grade_mean'] = df['loan_amount'] / df.groupby('grade_subgrade')['loan_amount'].transform('mean')
    df['income_vs_job_mean'] = df['annual_income'] / df.groupby('employment_status')['annual_income'].transform('mean')

    num_cols_to_transform = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    
    if 'loan_paid_back' in num_cols_to_transform:
        num_cols_to_transform.remove('loan_paid_back')
        
    for col in num_cols_to_transform:
        df[f'Log_{col}'] = np.log1p(df[col])
        
        df[f'{col}_sq'] = df[col]**2

    high_card_cols = ['annual_income', 'loan_amount']
    
    for c in high_card_cols:
        df[f'{c}_round'] = df[c].round(0).astype(str)
        df[f'{c}_thousands'] = df[c].round(-3).astype(str)

    return df

In [41]:
full_df = create_features(full_df)

# Split back into Train and Test
X = full_df.iloc[:train_len].copy()
X_test = full_df.iloc[train_len:].copy()

# Re-identify categorical columns (new bins created are categorical)
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical Features: {len(cat_cols)} variables")

Categorical Features: 10 variables


In [42]:
# ====================================================
# 4. ADVERSARIAL VALIDATION (Sanity Check)
# ====================================================
print("\n[3/6] Running Adversarial Validation Check...")
# We train a quick model to see if it can distinguish Train from Test.
# If AUC is ~0.5, Train and Test are similar (Good). If > 0.70, we have drift.

adv_train = X.copy()
adv_test = X_test.copy()
adv_train['is_test'] = 0
adv_test['is_test'] = 1
adv_data = pd.concat([adv_train, adv_test], axis=0).reset_index(drop=True)
adv_y = adv_data['is_test']
adv_X = adv_data.drop(columns=['is_test'])


[3/6] Running Adversarial Validation Check...


In [43]:
# Quick ordinal encoding for the check
adv_X_enc = ce.OrdinalEncoder(cols=cat_cols).fit_transform(adv_X)

adv_model = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1, 
                          tree_method='hist', device='cuda', eval_metric='auc', random_state=SEED)
                          
adv_cv_score = cross_val_score(adv_model, adv_X_enc, adv_y, cv=3, scoring='roc_auc').mean()

print(f"Adversarial AUC: {adv_cv_score:.4f}")
if adv_cv_score > 0.70:
    print("⚠️ WARNING: Train and Test sets look different. Consider dropping drifting features.")
else:
    print("✅ PASSED: Train and Test sets are similar. Proceeding...")

del adv_data, adv_X, adv_y, adv_X_enc, adv_train, adv_test
gc.collect()


Adversarial AUC: 0.4975
✅ PASSED: Train and Test sets are similar. Proceeding...


264

In [44]:
print(f"Starting {FOLDS}-Fold Stacking Ensemble...")

# Initialize arrays for Out-of-Fold (OOF) preds and Test preds
xgb_oof = np.zeros(len(X))
lgbm_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))

xgb_test = np.zeros(len(X_test))
lgbm_test = np.zeros(len(X_test))
cat_test = np.zeros(len(X_test))


Starting 10-Fold Stacking Ensemble...


In [45]:
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)

In [46]:
class KerasKaggleClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, input_dim=None, learning_rate=0.001, epochs=50, batch_size=1024, random_state=42):
        self.input_dim = input_dim
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.random_state = random_state
        self.model = None
        self.scaler = StandardScaler() # Neural Networks MUST be scaled

    def build_model(self, input_shape):
        tf.random.set_seed(self.random_state)
        model = Sequential([
            Input(shape=(input_shape,)),
            
            # Layer 1: Wide and Deep
            Dense(256),
            BatchNormalization(),
            Activation('swish'), # Swish often outperforms ReLU
            Dropout(0.3),
            
            # Layer 2
            Dense(128),
            BatchNormalization(),
            Activation('swish'),
            Dropout(0.2),
            
            # Layer 3
            Dense(64),
            BatchNormalization(),
            Activation('swish'),
            Dropout(0.1),
            
            # Output Layer
            Dense(1, activation='sigmoid')
        ])
        
        model.compile(
            optimizer=Adam(learning_rate=self.learning_rate),
            loss='binary_crossentropy',
            metrics=['AUC']
        )
        return model

    def fit(self, X, y, eval_set=None, **kwargs):
        # 1. Scale the data (Critical for NNs)
        X_scaled = self.scaler.fit_transform(X)
        
        # 2. Prepare Validation Data if present
        validation_data = None
        if eval_set:
            # eval_set is a list [(X_val, y_val)]
            X_val_raw, y_val = eval_set[0]
            X_val_scaled = self.scaler.transform(X_val_raw)
            validation_data = (X_val_scaled, y_val)
            
        # 3. Build & Train
        if self.model is None:
            self.model = self.build_model(X.shape[1])
            
        # Callbacks for smart training
        es = EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True, verbose=0)
        lr = ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=4, mode='max', min_lr=1e-6, verbose=0)
        
        self.model.fit(
            X_scaled, y,
            validation_data=validation_data,
            epochs=self.epochs,
            batch_size=self.batch_size,
            callbacks=[es, lr],
            verbose=0, # Silent training
            shuffle=True
        )
        return self

    def predict_proba(self, X):
        X_scaled = self.scaler.transform(X)
        preds = self.model.predict(X_scaled, batch_size=2048, verbose=0).flatten()
        # Sklearn expects (N, 2) output for binary classifier
        return np.vstack([1-preds, preds]).T

In [47]:
models = {
    # 1. XGBoost (Depth 6)
    'XGB6': XGBClassifier(
        learning_rate=0.01, n_estimators=5000, max_depth=6, subsample=0.92, 
        colsample_bytree=0.1, min_child_weight=3, lambda_=1.7, alpha=1.6,
        tree_method='hist', device='cuda', objective='binary:logistic', 
        eval_metric='auc', random_state=SEED, enable_categorical=True
    ),
    
    # 2. XGBoost (Depth 16 - Complex)
    'XGB_TE16': XGBClassifier(
        learning_rate=0.01, n_estimators=5000, max_depth=16, subsample=0.67,
        colsample_bytree=0.1, min_child_weight=2, lambda_=0.3, alpha=4.4,
        tree_method='hist', device='cuda', objective='binary:logistic', 
        eval_metric='auc', random_state=SEED, enable_categorical=True
    ),

    # 3. LightGBM
    'LGBM2': LGBMClassifier(
        learning_rate=0.03, n_estimators=5000, num_leaves=318, max_depth=5,
        subsample=0.47, colsample_bytree=0.22, reg_alpha=4.8, reg_lambda=0.01,
        objective='binary', metric='auc', device='gpu', random_state=SEED, verbose=-1
    ),

    # 4. CatBoost (GPU)
    'CAT': CatBoostClassifier(
        learning_rate=0.139, iterations=5000, depth=5, l2_leaf_reg=0.52,
        subsample=0.65, task_type='GPU', devices='0', eval_metric='AUC',
        random_seed=SEED, verbose=0, bootstrap_type='Bernoulli'
    ),
    
    # 5. NEW: Neural Network (Keras/GPU)
    'NN': KerasKaggleClassifier(
        learning_rate=0.005,
        epochs=60,
        batch_size=2048, 
        random_state=SEED
    )
}

print(f"Model Zoo Updated: {len(models)} models (All GPU-Accelerated).")

Model Zoo Updated: 5 models (All GPU-Accelerated).


In [ ]:
# The "Universal" Training Loop
print(f"Starting Ensemble Training for {len(models)} models...")

oof_preds_df = pd.DataFrame()
test_preds_df = pd.DataFrame()
scores = {}

# ... (Imports and Setup as before) ...

for name, model in models.items():
    print(f"\n🔹 Training Model: {name} ...")
    current_oof = np.zeros(len(X))
    current_test = np.zeros(len(X_test))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx].copy(), y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx].copy(), y.iloc[val_idx]
        X_te = X_test.copy()
        
        # Encoding logic
        if 'CAT' in name:
            pass # Raw data for CatBoost
        else:
            te_encoder = ce.TargetEncoder(cols=cat_cols, smoothing=20)
            X_tr = te_encoder.fit_transform(X_tr, y_tr)
            X_val = te_encoder.transform(X_val)
            X_te = te_encoder.transform(X_te)
        
        # FIT LOGIC UPDATED
        if 'NN' in name:
             # Neural Network uses eval_set for Early Stopping
             model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)])
             
        elif 'CAT' in name:
             model.fit(X_tr, y_tr, eval_set=(X_val, y_val), 
                       verbose=False, early_stopping_rounds=100, 
                       cat_features=cat_cols)
             
        elif 'XGB' in name:
             model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                       verbose=False, early_stopping_rounds=100)
             
        elif 'LGBM' in name:
             model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], 
                       eval_metric='auc', 
                       callbacks=[early_stopping(100, verbose=False)])

        # Predict & Accumulate (Same as before)
        val_preds = model.predict_proba(X_val)[:, 1]
        current_oof[val_idx] = val_preds
        current_test += model.predict_proba(X_te)[:, 1] / FOLDS

        
    model_score = roc_auc_score(y, current_oof)
    scores[name] = model_score
    print(f"   ✅ {name} Final CV AUC: {model_score:.5f}")
    
    oof_preds_df[name] = current_oof
    test_preds_df[name] = current_test
    
    pd.DataFrame({f'{name}': current_oof}).to_csv(f'oof_{name}.csv', index=False)
    pd.DataFrame({f'{name}': current_test}).to_csv(f'sub_{name}.csv', index=False)
    
    del X_tr, X_val, X_te, current_oof, current_test
    gc.collect()

Starting Ensemble Training for 5 models...

🔹 Training Model: XGB6 ...
   ✅ XGB6 Final CV AUC: 0.91063

🔹 Training Model: XGB_TE16 ...
   ✅ XGB_TE16 Final CV AUC: 0.91376

🔹 Training Model: LGBM2 ...
   ✅ LGBM2 Final CV AUC: 0.91190

🔹 Training Model: CAT ...


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


   ✅ CAT Final CV AUC: 0.92097

🔹 Training Model: NN ...


I0000 00:00:1764520891.154296      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15465 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1764520894.872407     943 service.cc:148] XLA service 0x7eaa1c013050 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1764520894.872467     943 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1764520895.252998     943 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1764520898.069420     943 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [ ]:
print("Training Meta-Model (Logistic Regression) on Model Zoo outputs...")

# The 'oof_preds_df' and 'test_preds_df' were automatically populated 
# by our new training loop in the previous step.

meta_model = LogisticRegression()
meta_model.fit(oof_preds_df, y)

stack_test_preds = meta_model.predict_proba(test_preds_df)[:, 1]

stack_ oof_preds = meta_model.predict_proba(oof_preds_df)[:, 1]
cv_score = roc_auc_score(y, stack_oof_preds)

print(f"\n---> 🏆 Final Stacked CV AUC: {cv_score:.5f}")

coefs = pd.Series(meta_model.coef_[0], index=oof_preds_df.columns)
print("\nMeta-Model Coefficients (Importance):")
print(coefs.sort_values(ascending=False))

In [ ]:
final_preds = np.clip(stack_test_preds, 0.001, 0.999)

submission[TARGET] = final_preds
submission.to_csv('submission.csv', index=False)

print("\n✅ Done! Predictions saved to 'submission.csv'")


In [ ]:
print(submission[TARGET].describe())

plt.figure(figsize=(8, 4))
sns.histplot(final_preds, kde=True, color='blue')
plt.title("Final Stacked Prediction Distribution")
plt.show()